# IntelliClaim — YOLOv8 Damage Detection Training

Train YOLOv8 on the CarDD dataset for vehicle damage detection.

**Setup:**
1. Upload your `cardd_yolo.zip` dataset to Kaggle as a dataset
2. Enable GPU accelerator (Settings → Accelerator → GPU T4 x2)
3. Run all cells
4. Download the best weights from the output

## 1. Install Dependencies

In [ ]:
!pip install -q ultralytics==8.4.77

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    vram = getattr(props, 'total_global_mem', None) or getattr(props, 'total_memory', 0)
    print(f"VRAM: {vram / 1024**3:.1f} GB")

## 2. Prepare Dataset

**Option A:** If you uploaded `cardd_yolo` as a Kaggle dataset, it will be at `/kaggle/input/cardd-yolo/`

**Option B:** Upload the zip and extract here

In [ ]:
import os
from pathlib import Path

# Adjust this path based on how you uploaded the dataset
# Option A: Kaggle dataset (read-only, need to copy)
KAGGLE_INPUT = Path("/kaggle/input/cardd-yolo")

# Option B: If you uploaded a zip file
# !unzip -q /kaggle/input/your-dataset-name/cardd_yolo.zip -d /kaggle/working/

# Working directory for training
WORK_DIR = Path("/kaggle/working/cardd_yolo")

if KAGGLE_INPUT.exists() and not WORK_DIR.exists():
    # Copy from read-only input to working dir
    !cp -r {KAGGLE_INPUT} {WORK_DIR}
    print(f"Copied dataset to {WORK_DIR}")
elif WORK_DIR.exists():
    print(f"Dataset already at {WORK_DIR}")
else:
    print("❌ Dataset not found! Upload cardd_yolo as a Kaggle dataset or zip.")

# Verify
print(f"\nTrain images: {len(list((WORK_DIR / 'images/train').glob('*.jpg')))}")
print(f"Val images:   {len(list((WORK_DIR / 'images/val').glob('*.jpg')))}")
print(f"Test images:  {len(list((WORK_DIR / 'images/test').glob('*.jpg')))}")

In [ ]:
# Create data.yaml with correct paths for Kaggle
data_yaml_content = f"""# CarDD Dataset — YOLOv8 config (Kaggle)
path: {WORK_DIR}
train: images/train
val: images/val
test: images/test

# Classes (6 damage categories)
nc: 6
names:
  0: dent
  1: scratch
  2: crack
  3: glass_shatter
  4: lamp_broken
  5: tire_flat
"""

data_yaml_path = WORK_DIR / "data.yaml"
data_yaml_path.write_text(data_yaml_content)
print(f"Written: {data_yaml_path}")
print(data_yaml_content)

## 3. Train YOLOv8

Key improvements over the previous training:
- **YOLOv8m** (medium) instead of nano/small — better feature extraction
- **imgsz=800** — larger input captures more detail
- **150 epochs** with patience=30 — more training time to converge
- **Stronger augmentation** — mosaic, mixup, scale variation
- **Lower initial LR** with cosine decay

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8-medium (better accuracy than small/nano)
model = YOLO("yolov8m.pt")

# Train with improved hyperparameters
results = model.train(
    data=str(data_yaml_path),
    epochs=150,
    batch=16,             # T4 can handle 16 at 800px for yolov8m
    imgsz=800,            # Larger input = better detail detection
    device=0,
    patience=30,          # Early stopping
    
    # Project output
    project="/kaggle/working/runs",
    name="cardd_yolov8m_v2",
    exist_ok=True,
    
    # Augmentation (aggressive for small dataset)
    hsv_h=0.02,           # Hue variation
    hsv_s=0.7,            # Saturation
    hsv_v=0.4,            # Brightness
    degrees=15.0,         # Rotation (accident photos vary in angle)
    translate=0.15,       # Translation
    scale=0.5,            # Scale variation (critical for multi-size damage)
    shear=5.0,            # Slight shear
    fliplr=0.5,           # Horizontal flip
    mosaic=1.0,           # Mosaic augmentation (full strength)
    mixup=0.15,           # Mixup (blend images)
    copy_paste=0.1,       # Copy-paste augmentation
    
    # Optimizer
    optimizer="AdamW",
    lr0=0.001,            # Lower initial LR for fine-tuning
    lrf=0.01,             # Final LR factor (cosine decay)
    weight_decay=0.0005,
    warmup_epochs=5,      # Longer warmup
    
    # Loss weights
    box=7.5,              # Box loss weight
    cls=0.5,              # Classification loss
    dfl=1.5,              # Distribution focal loss
    
    # Other
    close_mosaic=15,      # Disable mosaic last 15 epochs for fine-tuning
    verbose=True,
)

## 4. Evaluate Results

In [ ]:
# Validate on test set
best_model = YOLO("/kaggle/working/runs/cardd_yolov8m_v2/weights/best.pt")

metrics = best_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=800,
    conf=0.25,
    iou=0.5,
    verbose=True,
)

print(f"\n{'='*60}")
print(f"📊 Test Set Results")
print(f"{'='*60}")
print(f"  mAP@50:      {metrics.box.map50:.4f}")
print(f"  mAP@50-95:   {metrics.box.map:.4f}")
print(f"  Precision:   {metrics.box.mp:.4f}")
print(f"  Recall:      {metrics.box.mr:.4f}")
print(f"{'='*60}")

In [ ]:
# Per-class metrics
import pandas as pd

class_names = ["dent", "scratch", "crack", "glass_shatter", "lamp_broken", "tire_flat"]
per_class = pd.DataFrame({
    "Class": class_names,
    "AP@50": metrics.box.ap50,
    "AP@50-95": metrics.box.ap,
})
print(per_class.to_string(index=False))

## 5. Visual Inspection

In [ ]:
from IPython.display import Image, display
import glob

# Show training curves
results_img = "/kaggle/working/runs/cardd_yolov8m_v2/results.png"
if os.path.exists(results_img):
    display(Image(filename=results_img, width=900))

In [ ]:
# Run inference on a few test images
import random

test_images = list((WORK_DIR / "images/test").glob("*.jpg"))
sample_images = random.sample(test_images, min(6, len(test_images)))

results = best_model.predict(
    source=sample_images,
    conf=0.25,
    iou=0.5,
    imgsz=800,
    save=True,
    project="/kaggle/working/predictions",
    name="samples",
    exist_ok=True,
)

# Display predictions
pred_dir = "/kaggle/working/predictions/samples/"
for img_path in sorted(glob.glob(pred_dir + "*.jpg"))[:6]:
    display(Image(filename=img_path, width=600))
    print()

## 6. Export & Download

Download the `best.pt` file and place it at:
```
ml/weights/detection/cardd_yolov8/weights/best.pt
```

In [ ]:
import shutil

# Copy best weights to easy-to-find location
src = Path("/kaggle/working/runs/cardd_yolov8m_v2/weights/best.pt")
dst = Path("/kaggle/working/best_cardd_yolov8m.pt")
shutil.copy(src, dst)

# Also copy last.pt as backup
src_last = Path("/kaggle/working/runs/cardd_yolov8m_v2/weights/last.pt")
dst_last = Path("/kaggle/working/last_cardd_yolov8m.pt")
if src_last.exists():
    shutil.copy(src_last, dst_last)

print(f"✅ Best weights saved to: {dst}")
print(f"   File size: {dst.stat().st_size / 1024**2:.1f} MB")
print(f"\n📥 Download from Kaggle Output tab and place at:")
print(f"   ml/weights/detection/cardd_yolov8/weights/best.pt")

In [ ]:
# Optional: Export to ONNX for faster CPU inference
best_model.export(format="onnx", imgsz=800, simplify=True)
print("\n✅ ONNX export complete (for optional faster CPU inference)")

## 7. Compare with Previous Model (Optional)

If you have your old `best.pt`, upload it and compare.

In [ ]:
# Uncomment if you uploaded your old model for comparison
# old_model = YOLO("/kaggle/input/old-model/best.pt")
# old_metrics = old_model.val(data=str(data_yaml_path), split="test", imgsz=640, conf=0.25)
# 
# print(f"Old model  — mAP@50: {old_metrics.box.map50:.4f}")
# print(f"New model  — mAP@50: {metrics.box.map50:.4f}")
# print(f"Improvement: {(metrics.box.map50 - old_metrics.box.map50)*100:+.1f}%")